#  Tokenization in LLMs (LLaMA + Groq)

This notebook explores how Large Language Models (LLMs) process text using **tokens** instead of words.

We will:
- Convert text → tokens
- Inspect token IDs and decoded values
- Compare input vs output tokens
- Use Groq API for inference

## 📌 What are Tokens?

Tokens are the **basic units of text** that an LLM processes.

They are NOT exactly words. A token can be:
- A full word
- Part of a word
- A space + word
- Punctuation

Example:

"my name" → [" my", " name"]

spaces are included in tokens.

##  Tokenizing Model Output

We apply the same tokenizer to the model's response.

This allows us to:
- Compare input vs output tokens
- Measure token usage
- Understand response structure

In [1]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")

text = "Hi my name is DeepLIN and I like straberry smoothie"

tokens = tokenizer.encode(text)

print(tokens)

for t in tokens:
    print(f"{t} = {tokenizer.decode([t])}")

You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565 - if you loaded a llama tokenizer from a GGUF file you can ignore this message.


[1, 6324, 590, 1024, 338, 21784, 23714, 322, 306, 763, 5312, 16344, 10597, 347]
1 = <s>
6324 = Hi
590 = my
1024 = name
338 = is
21784 = Deep
23714 = LIN
322 = and
306 = I
763 = like
5312 = stra
16344 = berry
10597 = smooth
347 = ie


In [16]:
import os
from dotenv import load_dotenv
from groq import Groq
from transformers import AutoTokenizer

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")

prompt = "Hi my name is Deepu and I like strawberry smoothie"


input_tokens = tokenizer.encode(prompt)


res = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": prompt}]
)

output = res.choices[0].message.content


output_tokens = tokenizer.encode(output)

# counts
print("INPUT TOKEN COUNT:", len(input_tokens))
print("OUTPUT TOKEN COUNT:", len(output_tokens))


print("\n=== INPUT TOKENS ===")
for t in input_tokens:
    print(f"{t} -> '{tokenizer.decode([t])}'")

print("\n=== OUTPUT TOKENS ===")
for t in output_tokens:
    print(f"{t} -> '{tokenizer.decode([t])}'")

print("\n=== MODEL OUTPUT ===\n", output)

INPUT TOKEN COUNT: 15
OUTPUT TOKEN COUNT: 58

=== INPUT TOKENS ===
1 -> '<s>'
6324 -> 'Hi'
590 -> 'my'
1024 -> 'name'
338 -> 'is'
21784 -> 'Deep'
29884 -> 'u'
322 -> 'and'
306 -> 'I'
763 -> 'like'
380 -> 'st'
1610 -> 'raw'
16344 -> 'berry'
10597 -> 'smooth'
347 -> 'ie'

=== OUTPUT TOKENS ===
1 -> '<s>'
20103 -> 'Nice'
304 -> 'to'
5870 -> 'meet'
366 -> 'you'
29892 -> ','
21784 -> 'Deep'
29884 -> 'u'
29889 -> '.'
624 -> 'St'
1610 -> 'raw'
16344 -> 'berry'
10597 -> 'smooth'
347 -> 'ie'
338 -> 'is'
263 -> 'a'
628 -> 'del'
14803 -> 'icious'
7348 -> 'choice'
29889 -> '.'
1938 -> 'Do'
366 -> 'you'
763 -> 'like'
4417 -> 'adding'
738 -> 'any'
4266 -> 'special'
2348 -> 'ing'
1127 -> 'red'
10070 -> 'ients'
304 -> 'to'
596 -> 'your'
380 -> 'st'
1610 -> 'raw'
16344 -> 'berry'
10597 -> 'smooth'
347 -> 'ie'
29892 -> ','
470 -> 'or'
437 -> 'do'
366 -> 'you'
3013 -> 'keep'
372 -> 'it'
2560 -> 'simple'
411 -> 'with'
925 -> 'just'
380 -> 'st'
1610 -> 'raw'
495 -> 'ber'
2722 -> 'ries'
322 -> 'and'
5505 ->

In [17]:
import ollama
from transformers import AutoTokenizer

# tokenizer (LLaMA style)
tokenizer = AutoTokenizer.from_pretrained("hf-internal-testing/llama-tokenizer")

prompt = "Hi my name is Deepu and I like strawberry smoothie"


input_tokens = tokenizer.encode(prompt)

print("\n=== INPUT TOKEN COUNT ===")
print(len(input_tokens))

print("\n=== INPUT TOKENS ===")
for t in input_tokens:
    print(f"{t} -> '{tokenizer.decode([t])}'")

response = ollama.chat(
    model="llama3.2:1b",
    messages=[{"role": "user", "content": prompt}]
)

output = response["message"]["content"]

output_tokens = tokenizer.encode(output)

print("\n=== OUTPUT TOKEN COUNT ===")
print(len(output_tokens))

print("\n=== OUTPUT TOKENS ===")
for t in output_tokens:
    print(f"{t} -> '{tokenizer.decode([t])}'")

print("\n=== MODEL OUTPUT ===\n")
print(output)


=== INPUT TOKEN COUNT ===
15

=== INPUT TOKENS ===
1 -> '<s>'
6324 -> 'Hi'
590 -> 'my'
1024 -> 'name'
338 -> 'is'
21784 -> 'Deep'
29884 -> 'u'
322 -> 'and'
306 -> 'I'
763 -> 'like'
380 -> 'st'
1610 -> 'raw'
16344 -> 'berry'
10597 -> 'smooth'
347 -> 'ie'

=== OUTPUT TOKEN COUNT ===
73

=== OUTPUT TOKENS ===
1 -> '<s>'
20103 -> 'Nice'
304 -> 'to'
5870 -> 'meet'
366 -> 'you'
29892 -> ','
21784 -> 'Deep'
29884 -> 'u'
29889 -> '.'
739 -> 'It'
10083 -> 'sounds'
763 -> 'like'
366 -> 'you'
505 -> 'have'
263 -> 'a'
14225 -> 'sweet'
9758 -> 'spot'
363 -> 'for'
380 -> 'st'
1610 -> 'raw'
16344 -> 'berry'
10597 -> 'smooth'
583 -> 'ies'
29889 -> '.'
2688 -> 'They'
29915 -> '''
276 -> 're'
263 -> 'a'
5972 -> 'popular'
322 -> 'and'
2143 -> 'ref'
690 -> 'res'
2790 -> 'hing'
13748 -> 'drink'
393 -> 'that'
508 -> 'can'
367 -> 'be'
1407 -> 'very'
24064 -> 'satisfying'
29889 -> '.'
1938 -> 'Do'
366 -> 'you'
505 -> 'have'
263 -> 'a'
25448 -> 'favorite'
982 -> 'way'
310 -> 'of'
3907 -> 'making'
596 -> 'your